<a href="https://colab.research.google.com/github/juliocesardiaz/ibjjf_webscraping/blob/claude%2Fcreate-main-branch-TfsJi/IBJJF_web_scraping_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fake-useragent

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 5.4 MB/s eta 0:00:00


In [ ]:
# INSTALL DEPENDENCIES
!pip install beautifulsoup4 unidecode


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 7.8 MB/s eta 0:00:00


In [ ]:
# CODE BLOCK 1: THE SCOUT (FIXED)
import requests
from bs4 import BeautifulSoup
import pandas as pd
from fake_useragent import UserAgent
import re

# --- FIX: Ensure this URL string has NO brackets or markdown ---
INDEX_URL = "https://ibjjf.com/events/results"
MANIFEST_FILE = "ibjjf_manifest.csv"

def run_scout():
    print(f"🕵️ Scout launching against: {INDEX_URL}")
    ua = UserAgent()
    headers = {'User-Agent': ua.random}

    try:
        # Check for malformed URL before requesting
        if "[" in INDEX_URL or "]" in INDEX_URL:
            raise ValueError("The URL contains Markdown brackets. Please clean the INDEX_URL variable.")

        response = requests.get(INDEX_URL, headers=headers, timeout=15)
        response.raise_for_status()
    except Exception as e:
        print(f"❌ Critical Error contacting HQ: {e}")
        return None

    soup = BeautifulSoup(response.text, 'html.parser')
    events = []

    # Target specific result links with metadata attributes
    result_links = soup.find_all('a', class_='event-year-result')

    print(f"👀 Found {len(result_links)} result links. Extracting metadata...")

    for link in result_links:
        # 1. Extract Attributes (The fix for Event Names)
        event_name = link.get('data-n') # Gets "World Jiu-Jitsu Championship"
        year = link.get('data-y')
        url = link.get('href')

        if not url: continue

        # 2. Normalize URL
        if url.startswith("/"):
            url = f"https://ibjjf.com{url}"

        # 3. Classify Type
        if url.lower().endswith(".pdf"):
            etype = "PDF"
        elif "ibjjfdb.com" in url:
            etype = "MODERN_DB"
        else:
            etype = "LEGACY_HTML"

        # 4. Fallback for Year
        if not year:
            year_match = re.search(r'(19|20)\d{2}', link.get_text() + url)
            year = year_match.group(0) if year_match else "Unknown"

        events.append({
            "year": year,
            "event_name": event_name.strip() if event_name else "Unknown Event",
            "url": url,
            "type": etype
        })

    # Save to CSV
    if not events:
        print("⚠️ No events found. The selector might be mismatched.")
        return None

    df = pd.DataFrame(events)
    df = df.drop_duplicates(subset=['url'])
    df = df.sort_values(by='year', ascending=False)

    df.to_csv(MANIFEST_FILE, index=False)

    print(f"\n✅ Mission Complete.")
    print(f"📄 Manifest saved to: {MANIFEST_FILE}")
    print(f"🎯 Total Targets Found: {len(df)}")
    print("\n📊 Sample Data (Check Event Names):")
    print(df[['year', 'event_name']].head())

    return df

# Run it
df_manifest = run_scout()


🕵️ Scout launching against: https://ibjjf.com/events/results
👀 Found 1870 result links. Extracting metadata...

✅ Mission Complete.
📄 Manifest saved to: ibjjf_manifest.csv
🎯 Total Targets Found: 1863

📊 Sample Data (Check Event Names):
      year                                         event_name
636   2026  Rio Summer Kids International Open IBJJF Jiu-J...
50    2026              European Jiu-Jitsu IBJJF Championship
1524  2026  South American Jiu-Jitsu IBJJF Championship – ...
633   2026  Rio Summer International Open IBJJF Jiu-Jitsu ...
731   2026  San Jose Winter International Open IBJJF Jiu-J...


In [ ]:
# CODE BLOCK 2: THE HARVESTER
from google.colab import drive
import os
import requests
from fake_useragent import UserAgent
import pandas as pd
import time
import random
import hashlib

# 1. Mount Google Drive
print("💾 Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Configuration
BASE_DIR = "/content/drive/MyDrive/IBJJF_Data_Lake"
HTML_DIR = os.path.join(BASE_DIR, "html_raw")
PDF_DIR = os.path.join(BASE_DIR, "pdfs")

# Create directories if they don't exist
os.makedirs(HTML_DIR, exist_ok=True)
os.makedirs(PDF_DIR, exist_ok=True)

print(f"📂 Data Lake established at: {BASE_DIR}")

def sanitize_filename(text):
    """Makes a string safe for filenames."""
    text = str(text).replace(" ", "_")
    return "".join([c if c.isalnum() or c == "_" else "" for c in text])

def harvest_data_lake():
    # Load Manifest
    try:
        df = pd.read_csv("ibjjf_manifest.csv")
    except FileNotFoundError:
        print("❌ Manifest not found! Run Step 1 first.")
        return

    ua = UserAgent()
    total = len(df)

    print(f"🚜 Starting Harvest of {total} targets...")

    for index, row in df.iterrows():
        url = row['url']
        etype = row['type']
        year = str(row['year'])
        # Truncate long names to avoid filesystem errors
        event = str(row['event_name'])[:50]

        # Construct Unique Filename
        # Structure: Year_EventName_Hash.html
        safe_name = sanitize_filename(f"{year}_{event}")
        url_hash = hashlib.md5(url.encode()).hexdigest()[:6]

        if etype == "PDF":
            filename = f"{safe_name}_{url_hash}.pdf"
            save_path = os.path.join(PDF_DIR, filename)
        else:
            filename = f"{safe_name}_{url_hash}.html"
            save_path = os.path.join(HTML_DIR, filename)

        # CHECKPOINT: Skip if already downloaded
        if os.path.exists(save_path):
            if index % 10 == 0: print(f"⏭️ Skipping (Exists): {filename}")
            continue

        # DOWNLOAD
        print(f"⬇️ ({index+1}/{total}) Downloading: {filename} ...", end=" ")

        try:
            # Stealth Sleep (Random 2-5s)
            time.sleep(random.uniform(2, 5))

            headers = {'User-Agent': ua.random}
            response = requests.get(url, headers=headers, timeout=20)

            if response.status_code == 200:
                with open(save_path, 'wb') as f:
                    f.write(response.content)
                print("✅ OK")
            elif response.status_code == 403:
                print("⛔ 403 FORBIDDEN (Server blocked us)")
                print("⏳ Resting for 60 seconds...")
                time.sleep(60)
            else:
                print(f"⚠️ Status {response.status_code}")

        except Exception as e:
            print(f"❌ Error: {e}")

    print("\n🏁 Harvest Complete!")

# Run it
harvest_data_lake()


💾 Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Data Lake established at: /content/drive/MyDrive/IBJJF_Data_Lake
🚜 Starting Harvest of 1863 targets...
⏭️ Skipping (Exists): 2026_Rio_Summer_Kids_International_Open_IBJJF_JiuJitsu_3235f8.html
⏭️ Skipping (Exists): 2026_Miami_Winter_International_Open_IBJJF_JiuJitsu_No_022b42.html
⏭️ Skipping (Exists): 2025_Miami_International_Open_IBJJF_JiuJitsu_Champions_9392cd.html
⏭️ Skipping (Exists): 2025_São_Paulo_International_Open_IBJJF_JiuJitsu_Champ_6cdff5.html
⏭️ Skipping (Exists): 2025_New_York_Fall_International_Open_IBJJF_JiuJitsu_C_83320b.html
⏭️ Skipping (Exists): 2025_Sacramento_Fall_Kids_International_Open_IBJJF_Jiu_915a29.html
⏭️ Skipping (Exists): 2025_Curitiba_BJJ_Pro_IBJJF_Championship_be092b.html
⏭️ Skipping (Exists): 2025_American_National_Kids_IBJJF_JiuJitsu_Championshi_a19765.html
⏭️ Skipping (Exists): 2025_Pan_JiuJitsu_N

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import glob
import json
import re
from bs4 import BeautifulSoup
from unidecode import unidecode

# --- CONFIGURATION ---
INPUT_DIR = "/content/drive/MyDrive/IBJJF_Data_Lake/html_raw"
OUTPUT_BASE_DIR = "/content/drive/MyDrive/IBJJF_Data_Lake/structured_json"

# --- IDENTITY LOGIC ---
class BrazilianIdentityManager:
    def __init__(self):
        self.suffixes = {'junior', 'jr', 'jr.', 'filho', 'neto', 'sobrinho', 'ii', 'iii'}
        self.prefixes = ['da', 'de', 'do', 'das', 'dos', 'e']

    def parse_name(self, raw_name):
        if not raw_name: return None
        clean_text = unidecode(str(raw_name)).lower().strip()
        parts = clean_text.split()

        detected_suffix = None
        has_prefix = False

        if parts and parts[-1] in self.suffixes:
            detected_suffix = parts.pop()

        core_parts = []
        for p in parts:
            if p in self.prefixes: has_prefix = True
            core_parts.append(p)

        strict_id = "".join(core_parts)
        loose_parts = [p for p in core_parts if p not in self.prefixes]
        loose_id = "".join(loose_parts)

        return {
            "display_name": raw_name.strip(),
            "strict_id": strict_id,
            "loose_id": loose_id,
            "suffix": detected_suffix if detected_suffix else "",
            "prefix_flag": "Yes" if has_prefix else "No"
        }

# --- ORGANIZER LOGIC ---
class JsonOrganizer:
    def __init__(self):
        self.id_manager = BrazilianIdentityManager()

    def sanitize_folder_name(self, text):
        text = str(text).lower()
        text = text.replace("ibjjf", "").strip()
        text = re.sub(r'[^a-z0-9\s]', '', text)
        text = re.sub(r'\s+', '_', text)
        return text

    def _parse_category_string(self, cat_str):
        cat_str = re.sub(r'\(.*?\)', '', cat_str)
        parts = [p.strip() for p in cat_str.split('/')]
        data = {"age": "Unknown", "rank": "Unknown", "gender": "Unknown", "weight": "Unknown"}
        for p in parts:
            pl = p.lower()
            if pl in ['male', 'female']: data['gender'] = p
            elif any(b in pl for b in ['white', 'blue', 'purple', 'brown', 'black']): data['rank'] = p
            elif any(a in pl for a in ['adult', 'master', 'juvenile']): data['age'] = p
            else: data['weight'] = p
        return data

    def extract_metadata(self, soup, filename):
        parts = filename.split('_')
        year = parts[0] if parts[0].isdigit() else "Unknown"

        event_id = "Legacy"
        img_tag = soup.find('img', src=re.compile(r'Championship/Logo/\d+'))
        if img_tag:
            id_match = re.search(r'Logo/(\d+)', img_tag['src'])
            if id_match: event_id = id_match.group(1)

        date = "Unknown"
        status_tag = soup.find('small', class_='status')
        if status_tag:
            text = status_tag.get_text()
            date_match = re.search(r'[A-Za-z]{3}/\d{2}/\d{4}|\d{2}/\d{2}/\d{4}', text)
            if date_match: date = date_match.group(0)

        title_tag = soup.find('h2', class_='title')
        if title_tag:
            event_name = title_tag.get_text(strip=True)
        else:
            event_name = " ".join(parts[1:-1]).replace(".html", "")

        discipline = "No-Gi" if "no-gi" in event_name.lower() or "nogi" in event_name.lower() else "Gi"

        return {
            "event_name": event_name.strip(),
            "event_id": event_id,
            "year": year,
            "date": date,
            "discipline": discipline,
            "source_file": filename
        }

    def process_file(self, filepath):
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                html = f.read()
            soup = BeautifulSoup(html, 'html.parser')
            filename = os.path.basename(filepath)

            meta = self.extract_metadata(soup, filename)
            athletes = []

            if soup.find('h4', class_='subtitle'):
                athletes = self._scrape_modern(soup)
            elif soup.find('div', class_='category'):
                athletes = self._scrape_legacy(soup)

            if not athletes:
                return

            self._save_json(meta, athletes)

        except Exception as e:
            print(f"❌ Error processing {filepath}: {e}")

    def _scrape_modern(self, soup):
        results = []
        for cat_header in soup.find_all('h4', class_='subtitle'):
            cat_str = cat_header.get_text(strip=True)
            list_div = cat_header.find_next_sibling('div', class_='list')
            if not list_div: continue

            athlete_items = list_div.find_all('div', class_='athlete-item')
            if not athlete_items: continue

            cat_info = self._parse_category_string(cat_str)

            for item in athlete_items:
                place_tag = item.find('div', class_='position-athlete')
                place = place_tag.get_text(strip=True) if place_tag else "N/A"

                name_div = item.find('div', class_='name')
                if name_div and name_div.p:
                    span = name_div.p.find('span')
                    team = span.get_text(strip=True) if span else "Unknown"
                    if span: span.extract()
                    name = name_div.p.get_text(strip=True)

                    ident = self.id_manager.parse_name(name)
                    if ident:
                        results.append({
                            "name": ident['display_name'],
                            "team": team,
                            "place": place,
                            "ids": ident,
                            "category": cat_info
                        })
        return results

    def _scrape_legacy(self, soup):
        results = []
        for cat_div in soup.find_all('div', class_='category'):
            cat_str = cat_div.get_text(strip=True)
            cat_info = self._parse_category_string(cat_str)
            table = cat_div.find_next_sibling('table')
            if not table: continue

            for row in table.find_all('tr'):
                cols = row.find_all('td')
                if not cols: continue
                place = cols[0].get_text(strip=True)

                details = cols[1]
                n_div = details.find('div', class_='athlete-name')
                t_div = details.find('div', class_='academy-name')

                name = n_div.get_text(strip=True) if n_div else "Unknown"
                team = t_div.get_text(strip=True) if t_div else "Unknown"

                ident = self.id_manager.parse_name(name)
                if ident:
                    results.append({
                        "name": ident['display_name'],
                        "team": team,
                        "place": place,
                        "ids": ident,
                        "category": cat_info
                    })
        return results

    def _save_json(self, meta, athletes):
        clean_name = self.sanitize_folder_name(meta['event_name'])
        year = meta['year']

        target_dir = os.path.join(OUTPUT_BASE_DIR, clean_name, year)
        os.makedirs(target_dir, exist_ok=True)

        # --- NEW FILENAME LOGIC ---
        # The JSON file is now dynamically named after the event and year
        json_filename = f"{clean_name}_{year}.json"
        output_path = os.path.join(target_dir, json_filename)

        final_data = {
            "tournament_name": meta['event_name'],
            "tournament_year": year,
            "metadata": meta,
            "total_athletes": len(athletes),
            "athletes": athletes
        }

        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(final_data, f, indent=2, ensure_ascii=False)

# --- EXECUTION ---
organizer = JsonOrganizer()
files = glob.glob(os.path.join(INPUT_DIR, "*.html"))

print(f"📂 Reorganizing {len(files)} files into structured JSON folders...")

for i, f in enumerate(files):
    organizer.process_file(f)
    if i % 20 == 0: print(f"  Processed {i} files...")

print(f"\n✨ Organization Complete! Check folders in: {OUTPUT_BASE_DIR}")


📂 Reorganizing 1863 files into structured JSON folders...
  Processed 0 files...
  Processed 20 files...
  Processed 40 files...
  Processed 60 files...
  Processed 80 files...
  Processed 100 files...
  Processed 120 files...
  Processed 140 files...
  Processed 160 files...
  Processed 180 files...
  Processed 200 files...
  Processed 220 files...
  Processed 240 files...
  Processed 260 files...
  Processed 280 files...
  Processed 300 files...
  Processed 320 files...
  Processed 340 files...
  Processed 360 files...
  Processed 380 files...
  Processed 400 files...
  Processed 420 files...
  Processed 440 files...
  Processed 460 files...
  Processed 480 files...
  Processed 500 files...
  Processed 520 files...
  Processed 540 files...
  Processed 560 files...
  Processed 580 files...
  Processed 600 files...
  Processed 620 files...
  Processed 640 files...
  Processed 660 files...
  Processed 680 files...
  Processed 700 files...
  Processed 720 files...
  Processed 740 files..